In [2]:
library(showtext)
font_add("Arial", "/System/Library/Fonts/Supplemental/Arial.ttf")
showtext_auto()

Loading required package: sysfonts

Loading required package: showtextdb



In [172]:
library(tidyverse)
library(ggplot2)
library(ggpubr)
library(readr)
library(stringr)
library(colorBlindness)
library(Statial)
library(rstatix)
library(ComplexHeatmap)
library(circlize)
library(spatstat.geom)
library(spatstat.model)
library(jsonlite)
library(reticulate)
library(data.table)
library(scales)
library(data.table)
library(cowplot)
library(patchwork)


Attaching package: ‘ggpubr’


The following object is masked from ‘package:cowplot’:

    get_legend


The following objects are masked from ‘package:spatstat.geom’:

    border, rotate




## Results dir

In [4]:
results_dir = "/Users/jawadalaaedeen/Desktop/PhD/NMC/results"

# Palette

In [162]:
myeloid_palette <- c(
  "MDSCs" = "#332288",
  "M2 macrophages" = "#66A61E",
  "M1 macrophages" = "#00557F",
  "M1/M2 macrophages" = "#BBCCEE",
  "DCs" = "#E69F00",
  "Granulocytes" = "#B85AA6",
  "Mast cells" = "#8E006A"
)

lymphoid_palette <- c(
  "CD8+ T cells" = "#B2182B",
  "CD4+ T cells" = "#EF8A62",
  "Treg T cells" = "#FFCCCC",
  "NK cells" = "#0096FF",
  "B cells" = "#E6C84F",
  "PCs" = "#F4E58A"
)

non_immune_palette <- c(
  "Actin+ cells" = "#B39DDB",
  "Endothelial cells" = "#7F8F6A",
  "Lymphatic endothelial cells" = "#A6B08A",
  "Epithelial cells" = "#E3B07A"
)


other_palette <- c(
  "Tumor cells" = "#44AA99",
  "NFC" = "#DDDDDD"
)

full_palette <- c(
  myeloid_palette,
  lymphoid_palette,
  non_immune_palette,
  other_palette
)


tumor_site_colors <- c(
"Lung" = "#F0F0F0",     # very light
"Head and Neck" = "#B0B0B0",  # medium
"Extrahepatic" = "#505050" # darker
)

# Loading Primary dataset

In [200]:
celltype_metadata <- readr::read_csv(
  paste0(results_dir, "/celltype_metadata_final.csv")
) %>%
  filter(!patient_exp %in% c("NMC_6_ROI6", "NMC_12_ROI6", "NMC_21_ROI23")) %>%
  mutate(is_lung = ifelse(tumor_site == "Lung", "Lung", "Head and Neck/Extrahepatic")) %>%
  mutate("patient_ID" = str_replace(patient_ID, "NMC", "NUT"),
         "patient_exp" = str_replace(patient_exp, "NMC", "NUT"))

#patient exp are written basically like this: NUT_2_ROI3 etc.. let's arrange them properly so based on patient number and then ROI number and extract that order
celltype_metadata <- celltype_metadata %>%
  mutate(patient_num = as.numeric(str_extract(patient_exp, "(?<=NUT_)\\d+")),
         roi_num = as.numeric(str_extract(patient_exp, "(?<=ROI)\\d+"))) %>%
  arrange(patient_num, roi_num) %>%
  select(-patient_num, -roi_num)
celltype_metadata$patient_exp <- factor(celltype_metadata$patient_exp, levels = unique(celltype_metadata$patient_exp))
celltype_metadata$patient_ID <- factor(celltype_metadata$patient_ID, levels = unique(celltype_metadata$patient_ID))
patient_ID_order <- unique(celltype_metadata$patient_ID)
patient_exp_order <- unique(celltype_metadata$patient_exp)


myeloid <-  c("M1 macrophages", "M2 macrophages", "M1/M2 macrophages", "DCs", "Granulocytes", "MDSCs", "Mast cells")
lymphoid <- c("CD4+ T cells", "CD8+ T cells", "Treg T cells", "NK cells", "B cells", "PCs")

Rows: 578608 Columns: 16
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (9): cell_type, cell_category, patient_ID, exp_name, patient_exp, sample...
dbl (6): cell_id, Cell Center X, Cell Center Y, run, rois, survival_time_months
lgl (1): is_sample_primary

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [210]:
survival_df <- read.csv(
    paste0(results_dir, "/survival_data.csv")
) %>%
#add a factor order for tumor_site column
mutate(tumor_site = factor(tumor_site, levels = c("Lung", "Head and Neck", "Extrahepatic")),
patient_ID = factor(patient_ID, levels = patient_ID_order),
is_lung = ifelse(tumor_site == "Lung", "Lung", "Head and Neck/Extrahepatic"))
survival_df

patient_ID,survival_time_months,tumor_site,event,is_lung
<fct>,<dbl>,<fct>,<int>,<chr>
NUT_1,170,Head and Neck,0,Head and Neck/Extrahepatic
NUT_2,3,Lung,1,Lung
NUT_4,4,Lung,1,Lung
NUT_5,9,Head and Neck,1,Head and Neck/Extrahepatic
NUT_6,7,Lung,1,Lung
NUT_7,10,Lung,1,Lung
NUT_8,4,Lung,1,Lung
NUT_9,2,Lung,1,Lung
NUT_12,48,Head and Neck,1,Head and Neck/Extrahepatic


# Figure 1

## A - Cell type proportions across ROIs

In [ ]:
#showcase proportions of celltypes per patient_exp in a stacked barplot
celltype_proportions_exp <- celltype_metadata %>%
  select(patient_exp, cell_category) %>%
  group_by(patient_exp, cell_category) %>%
  summarise(cell_count = n()) %>%
  ungroup() %>%
  group_by(patient_exp) %>%
  mutate(percentage = (cell_count / sum(cell_count)) * 100) %>%
  ungroup()
celltype_proportions_exp

In [ ]:
celltype_proportions_exp_plot = ggplot(
  celltype_proportions_exp,
  aes(x = patient_exp, y = percentage, fill = cell_category)
) +
  geom_bar(stat = "identity", width = 0.85) +
  scale_fill_manual(values = full_palette) +
  guides(fill = guide_legend(reverse = TRUE)) +
  labs(
    x = "Patient ROI",
    y = "Percentage of cells (%)",
    fill = "Cell type"
  ) +
  scale_y_continuous(limits = c(0, 105), expand = c(0, 0), sec.axis = dup_axis()) +
  theme_minimal(base_size = 13) +
  theme(
    axis.text.y = element_text(size = 20, colour = "black"),
    axis.text.x = element_text(size = 20, colour = "black", angle = 45, hjust = 1),
    axis.title.y = element_text(size = 20, colour = "black"),
    axis.title.x = element_text(size = 20, colour = "black", margin = margin(t = 10)),
    panel.border = element_blank(),
    axis.line.x.bottom = element_line(colour="black", linewidth=0.6),
    axis.line.x.top    = element_line(colour="black", linewidth=0.6),
    axis.line.y.left   = element_line(colour="black", linewidth=0.6),
    axis.line.y.right  = element_blank(),
    axis.text.x.top  = element_blank(),
    axis.ticks.x.top = element_blank(),
    axis.title.x.top = element_blank(),
    axis.ticks.y.right = element_blank(),
    axis.title.y.right = element_blank(),
    axis.text.y.right = element_blank(),
    panel.grid = element_blank(),
    legend.position = "none",
  )
celltype_proportions_exp_plot
ggsave(filename = "/Users/jawadalaaedeen/Desktop/images_for_figures/supp_fig1/celltype_proportions_barplot.pdf",
       plot = celltype_proportions_exp_plot,
       width = 12,
       height = 7,
       dpi = 300)

## B- Pearson Correlation of cell types CLRs across ROIs

In [134]:
#calculate celltype_proportions per patient_exp 
celltype_counts_exp <- celltype_metadata %>%
select(patient_exp, cell_category) %>%
group_by(patient_exp, cell_category) %>%
summarise(cell_count = n()) %>%
ungroup()

#Pivot wider for CLR
celltype_counts_wide <- celltype_counts_exp %>%
  pivot_wider(
    names_from = cell_category,
    values_from = cell_count,
    values_fill = 0
  ) %>%
  mutate(across(where(is.numeric), ~ .x + 1))

#Apply CLR per patient_exp
celltype_counts_clr <- celltype_counts_wide %>%
  rowwise() %>%
  mutate(across(where(is.numeric), ~ log(.x / exp(mean(log(c_across(where(is.numeric))))))))

#compute correlations between patient_exps based on their celltypes CLRs
celltype_clr_matrix <- celltype_counts_clr %>%
  select(-patient_exp) %>%
  as.matrix()
cor_matrix <- cor(t(celltype_clr_matrix), method = "pearson")
#add the patient_exp names as row and column names to the correlation matrix
rownames(cor_matrix) <- celltype_counts_clr$patient_exp
colnames(cor_matrix) <- celltype_counts_clr$patient_exp
#make self correlations NA
diag(cor_matrix) <- NA
cor_matrix

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_exp and cell_category.
ℹ Output is grouped by patient_exp.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_exp, cell_category))` for per-operation
  grouping (`?dplyr::dplyr_by`) instead.


,NUT_1_ROI1,NUT_1_ROI3,NUT_2_ROI1,NUT_4_ROI9,NUT_5_ROI10,NUT_5_ROI11,NUT_6_ROI5,NUT_7_ROI13,NUT_7_ROI14,NUT_8_ROI17,⋯,NUT_9_ROI4,NUT_12_ROI5,NUT_13_ROI14,NUT_13_ROI15,NUT_14_ROI8,NUT_14_ROI11,NUT_15_ROI7,NUT_15_ROI16,NUT_16_ROI17,NUT_16_ROI19
NUT_1_ROI1,NA,0.9278618,0.5366316,0.6159844,0.3995893,0.4317903,0.31357847,0.78409591,0.7720869,0.6888469,⋯,0.66051546,0.7325922,0.8381999,0.8090916,0.7691388,0.6661846,0.5539249,0.4757684,0.62508895,0.67348699
NUT_1_ROI3,0.9278618,NA,0.5397242,0.6356673,0.4048268,0.4355987,0.14175666,0.76502861,0.7282708,0.6189781,⋯,0.63601209,0.7640067,0.8791523,0.8150600,0.7924158,0.7446298,0.5561372,0.4721678,0.60506162,0.77583716
NUT_2_ROI1,0.5366316,0.5397242,NA,0.8010099,0.8389304,0.8868251,0.31774510,0.73529080,0.6596090,0.6506291,⋯,0.25841059,0.3613751,0.4751184,0.5094683,0.6391324,0.5306663,0.7615979,0.7884198,0.36816646,0.69191931
NUT_4_ROI9,0.6159844,0.6356673,0.8010099,NA,0.6665754,0.7055905,0.12558060,0.84210568,0.7475235,0.8580608,⋯,0.60323185,0.5812358,0.5119831,0.5572473,0.7921358,0.6602206,0.6556145,0.5950287,0.67643608,0.82491654
NUT_5_ROI10,0.3995893,0.4048268,0.8389304,0.6665754,NA,0.8886405,0.33351016,0.48744396,0.4957830,0.5634838,⋯,0.29208947,0.3100852,0.4516715,0.4857956,0.4627455,0.4879582,0.8300073,0.7407469,0.38913284,0.53897124
NUT_5_ROI11,0.4317903,0.4355987,0.8868251,0.7055905,0.8886405,NA,0.24120976,0.64076938,0.5878080,0.5716557,⋯,0.24505114,0.2371057,0.4291548,0.4581708,0.4244141,0.3924431,0.7782424,0.7369628,0.35556588,0.51179625
NUT_6_ROI5,0.3135785,0.1417567,0.3177451,0.1255806,0.3335102,0.2412098,NA,0.08787252,0.1535619,0.1684415,⋯,0.09437449,0.3981252,0.2171775,0.2618344,0.3449783,0.3165376,0.3393832,0.4572667,0.08153907,0.05658586
NUT_7_ROI13,0.7840959,0.7650286,0.7352908,0.8421057,0.4874440,0.6407694,0.08787252,NA,0.9041390,0.8162254,⋯,0.61401566,0.5804686,0.7004304,0.7396899,0.8242901,0.7046066,0.5463669,0.5006901,0.70550196,0.78114270
NUT_7_ROI14,0.7720869,0.7282708,0.6596090,0.7475235,0.4957830,0.5878080,0.15356190,0.90413895,NA,0.8328203,⋯,0.59730508,0.6464713,0.6788966,0.7856432,0.8108047,0.7187400,0.4817471,0.4021539,0.69840047,0.72723885
NUT_8_ROI17,0.6888469,0.6189781,0.6506291,0.8580608,0.5634838,0.5716557,0.16844146,0.81622535,0.8328203,NA,⋯,0.74671186,0.5257919,0.6038625,0.7007452,0.7656096,0.6885292,0.5906120,0.4735047,0.82221619,0.75703823


In [135]:
#Build a heatmap of the correlation matrix using ComplexHeatmap
cor_matrix_main_ht <- Heatmap(cor_matrix, 
        name = "Pearson Correlation", 
        show_row_names = TRUE, 
        col = colorRamp2(c(0, 0.5, 1), c("blue", "white", "red")),
        show_column_names = TRUE,
        cluster_rows = FALSE,
        cluster_columns = FALSE,
        row_names_gp = gpar(fontsize = 13),
        column_names_gp = gpar(fontsize = 13))
pdf("/Users/jawadalaaedeen/Desktop/images_for_figures/supp_fig1/correlation_heatmap.pdf", width = 8, height = 6)
draw(cor_matrix_main_ht)
dev.off()


agg_record_1476160405 
                    2

In [136]:
#for cor_matrix, make every value NA except the ones whos column name and rowname has the same "NUT_"
cor_matrix_same_patient <- cor_matrix
for (i in 1:nrow(cor_matrix_same_patient)) {
  for (j in 1:ncol(cor_matrix_same_patient)) {
    if (str_extract(rownames(cor_matrix_same_patient)[i], "(?<=NUT_)\\d+") != str_extract(colnames(cor_matrix_same_patient)[j], "(?<=NUT_)\\d+")) {
      cor_matrix_same_patient[i, j] <- NA
    }
  }
}
cor_matrix_same_patient

,NUT_1_ROI1,NUT_1_ROI3,NUT_2_ROI1,NUT_4_ROI9,NUT_5_ROI10,NUT_5_ROI11,NUT_6_ROI5,NUT_7_ROI13,NUT_7_ROI14,NUT_8_ROI17,⋯,NUT_9_ROI4,NUT_12_ROI5,NUT_13_ROI14,NUT_13_ROI15,NUT_14_ROI8,NUT_14_ROI11,NUT_15_ROI7,NUT_15_ROI16,NUT_16_ROI17,NUT_16_ROI19
NUT_1_ROI1,NA,0.9278618,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_1_ROI3,0.9278618,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_2_ROI1,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_4_ROI9,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_5_ROI10,NA,NA,NA,NA,NA,0.8886405,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_5_ROI11,NA,NA,NA,NA,0.8886405,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_6_ROI5,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_7_ROI13,NA,NA,NA,NA,NA,NA,NA,NA,0.904139,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_7_ROI14,NA,NA,NA,NA,NA,NA,NA,0.904139,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA
NUT_8_ROI17,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA,⋯,NA,NA,NA,NA,NA,NA,NA,NA,NA,NA


In [137]:
#Build a heatmap of the correlation matrix using ComplexHeatmap
cor_matrix_sub_ht <- Heatmap(cor_matrix_same_patient, 
        name = "Pearson Correlation", 
        #add a color palette that goes from white to red for the values of the correlation matrix, with white being 0, 0.5 and red being 1
        col = colorRamp2(c(0, 0.5, 1), c("blue", "white", "red")),
        show_row_names = TRUE, 
        show_column_names = TRUE,
        cluster_rows = FALSE,
        cluster_columns = FALSE,
        row_names_gp = gpar(fontsize = 13),
        column_names_gp = gpar(fontsize = 13))
pdf("/Users/jawadalaaedeen/Desktop/images_for_figures/supp_fig1/correlation_heatmap_same_patient.pdf", width = 8, height = 6)
draw(cor_matrix_sub_ht)
dev.off()

agg_record_304546228 
                   2

## C. Correlations only within the two ROIs of the same patient

In [104]:
#turn this into a long df: cor_matrix_same_patient
cor_matrix_same_patient_long <- as.data.frame(as.table(cor_matrix_same_patient)) %>%
  rename(patient_exp1 = Var1, patient_exp2 = Var2, correlation = Freq) %>%
  filter(!is.na(correlation)) %>%
  #extract patient_ID
    mutate(patient_ID = str_extract(patient_exp1, "NUT_\\d+")) %>%
    select(patient_ID, correlation) %>%
    distinct()
cor_matrix_same_patient_long

patient_ID,correlation
<chr>,<dbl>
NUT_1,0.9278618
NUT_5,0.8886405
NUT_7,0.9041390
NUT_9,0.9110020
NUT_13,0.9478431
NUT_14,0.9118550
NUT_15,0.9386723
NUT_16,0.7426154


In [153]:
#boxplot of cor_matrix_same_patient_long with all the values combined (so one box plot for the plot)
correlation_boxplot <- ggplot(cor_matrix_same_patient_long, aes(x = "", y = correlation)) +
  geom_boxplot(fill = "#44AA99", color = "black") +
  geom_jitter(width = 0.1, color = "black", size = 2) +
  theme_minimal() +
  labs(x = "Patient with two ROIs", y = "Pearson's correlation between the two ROIs") +
  theme(axis.text.x = element_text(color = "black", size = 13),
    axis.title.x = element_text(color = "black", size = 13, margin = margin(t = 10)),
    axis.title.y = element_text(color = "black", size = 13, margin = margin(r = 10)),
    axis.text.y = element_text(color = "black", size = 13),
    plot.title = element_text(color = "black", size = 16, hjust = 0.5, margin = margin(b = 20)),
    axis.line.x.bottom = element_line(color = "black", size = 0.8),
    axis.line.y.left = element_line(color = "black", size = 0.8))


#ggsave
ggsave(
  filename = "/Users/jawadalaaedeen/Desktop/images_for_figures/supp_fig1/correlation_boxplot.pdf",
  plot = correlation_boxplot,
  width = 3,
  height = 5,
  dpi = 300
)

# Figure 2

In [191]:
celltype_proportions_patient <- celltype_proportions_exp %>%
mutate(patient_ID = str_extract(patient_exp, "NUT_\\d+")) %>%
group_by(patient_ID, cell_category) %>%
summarise(percentage = median(percentage)) %>%
#left join to them the tumor site and is_lung from celltype_metadata
left_join(
  celltype_metadata %>%
    select(patient_ID, tumor_site, is_lung) %>%
    distinct(),
  by = "patient_ID"
) %>%
ungroup() %>%
filter(cell_category %in% c(lymphoid, myeloid))
celltype_proportions_patient_nohep <- celltype_proportions_patient %>%
filter(tumor_site != "Extrahepatic")

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_ID and cell_category.
ℹ Output is grouped by patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_ID, cell_category))` for per-operation
  grouping (`?dplyr::dplyr_by`) instead.


In [198]:
pd <- position_dodge(width = 0.8)


proportions_sites_plot <- ggplot(celltype_proportions_patient_nohep, aes(x = tumor_site, y = percentage, fill = tumor_site)) +
  geom_boxplot(
    position = pd,
    width = 0.6,
    alpha = 0.6,
    color = "black",
    outlier.shape = NA
  ) +
  geom_jitter(
    position = position_jitterdodge(jitter.width = 0.15, dodge.width = 0.8),
    size = 1.5,
    alpha = 0.7,
    inherit.aes = TRUE
  ) + 
  stat_compare_means(
    method = "wilcox.test",
    label = "p.format",
    paired = FALSE,
    label.y.npc = 0.75,
    aes(group = tumor_site),
    size = 3) +
  scale_fill_manual(values = tumor_site_colors) +
  scale_color_manual(values = tumor_site_colors) +
  labs(x = "Tumor tissue of origin", y = "Percentage") +
  theme_classic() +
    theme(
    legend.position = "none",
    axis.text.x = element_text(angle = 45, hjust = 1, size = 13),
    axis.title.x = element_text(size = 13, colour = "black", margin = margin(t = 10)),
    axis.text.y = element_text(size = 13, colour = "black"),
    axis.title.y = element_text(size = 13, colour = "black", margin = margin(r = 10)),
    axis.line = element_line(linewidth = 0.5, colour = "black"),
    axis.ticks = element_line(linewidth = 0.5, colour = "black"),
    strip.text = element_text(size = 13)
  ) +
  facet_wrap(~ cell_category, scales = "free_y") 

ggsave(
  filename = "/Users/jawadalaaedeen/Desktop/images_for_figures/supp_fig2/proportions_sites_boxplot.pdf",
  plot = proportions_sites_plot,
  width = 12,
  height = 7,
  dpi = 300
)

Warning message:
“No shared levels found between `names(values)` of the manual scale and the
data's colour values.”
